# Crew Spend Tracker — shared wallet set guards

Three agents share one **wallet set** with set-level **rate limit** and **budget** guards.

**Prereq:** `pip install -e "./python[dev]" jupyter`

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))
from _notebook_utils import FAKE_RECIPIENT, enable_mock_router, display_ledger

from algopay import AlgoPay
from algopay.core.types import Network

AGENTS = [
    {"role": "researcher", "calls": 3},
    {"role": "writer", "calls": 2},
    {"role": "reviewer", "calls": 4},
]

client = AlgoPay(network=Network.ALGORAND_TESTNET)
enable_mock_router(client)

In [ ]:
ws = client.wallet.create_wallet_set("crew-alpha")
await client.add_rate_limit_guard_for_set(ws.id, max_per_minute=5, name="crew_rpm")
await client.add_budget_guard_for_set(ws.id, daily_limit="0.25", name="crew_daily")

wallets = {}
for agent in AGENTS:
    w = client.wallet.create_wallet(ws.id, name=agent["role"])
    wallets[agent["role"]] = w
    print(f"{agent['role']:10} wallet {w.id[:8]}...")

In [ ]:
log = []
for agent in AGENTS:
    w = wallets[agent["role"]]
    for i in range(agent["calls"]):
        purpose = f"{agent['role']} task #{i + 1}"
        r = await client.pay(w.id, FAKE_RECIPIENT, "0.01", wallet_set_id=ws.id, purpose=purpose)
        log.append({"agent": agent["role"], "call": i + 1, "status": "ok" if r.success else r.status.value})

try:
    import pandas as pd
    from IPython.display import display
    display(pd.DataFrame(log))
except ImportError:
    for row in log:
        print(row)

In [ ]:
entries = []
for w in wallets.values():
    entries.extend(await client.ledger.query(wallet_id=w.id, limit=50))

completed = [e for e in entries if e.status.value == "completed"]
total = sum(e.amount for e in completed)
print(f"Crew ledger: {len(entries)} entries, {total} USDC completed")
display_ledger(entries)